<a href="https://colab.research.google.com/github/Suchrahmadhani/DataScience-Practicum/blob/main/Pertemuan3_Suci_Rahmadhani_240401070508.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 3: Data Cleaning, Missing Values, Outlier & Ekstraksi Data

## Identitas Mahasiswa

| Keterangan | Informasi |
|------------|-----------|
| **Nama** | Suci Rahmadhani |
| **NIM** | 240401070508 |
| **Kelas** | IF404 |
| **Program Studi** | PJJ Informatika |
| **Mata Kuliah** | Data Science |

# Tujuan Praktikum

Pada praktikum ini saya mempelajari proses data cleaning sebagai salah satu tahapan penting dalam Data Science. Praktikum meliputi proses mendeteksi missing values, menghapus data duplikat, melakukan normalisasi data, menangani missing values menggunakan teknik imputasi, mendeteksi dan menangani outlier menggunakan metode IQR Fence, serta mengambil data dari REST API menggunakan Python.

# Ringkasan Materi

Data cleaning merupakan proses memperbaiki kualitas data sebelum dilakukan analisis. Dataset yang belum dibersihkan biasanya memiliki berbagai permasalahan seperti missing values, data duplikat, inkonsistensi penulisan, maupun outlier.

Pada praktikum ini digunakan library Pandas untuk memproses dataset, NumPy untuk komputasi numerik, serta Requests untuk mengambil data dari REST API. Setelah proses pembersihan selesai, dataset divalidasi dan disimpan kembali sebagai dataset baru yang siap digunakan untuk analisis lebih lanjut.

In [23]:
import pandas as pd
import numpy as np
import requests

# Eksplorasi Awal Dataset

In [2]:
df = pd.read_csv("housing_dirty.csv")

print("Lima data pertama")
df.head()

Lima data pertama


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
0,1,297.0,1084.0,jogja,2.0,2000,baik
1,2,254.0,761.0,Medan,NaN,1995,Bagus
2,3,249.7,895.0,Depok,NaN,1983,baik
3,4,49.7,178.0,YGY,5.0,2013,baik
4,5,133.4,424.0,Medan,5.0,2004,Sedang


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB


In [4]:
df.describe()

,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,112.000000,1.130000e+02,120.000000,130.000000
mean,65.500000,267.627679,8.856325e+05,3.433333,2062.638462
std,37.671829,885.664181,9.407144e+06,1.776283,701.684043
min,1.000000,-50.000000,-5.000000e+02,1.000000,1890.000000
25%,33.250000,87.050000,3.450000e+02,2.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,4.000000,2002.000000
75%,97.750000,280.675000,9.550000e+02,5.000000,2011.750000
max,130.000000,9500.000000,1.000000e+08,6.000000,9999.000000


In [5]:
print("Ukuran Dataset :", df.shape)

Ukuran Dataset : (130, 7)


In [6]:
# Mengecek missing values

df.isnull().sum()

,0
id,0
luas_m2,18
harga_juta,17
kota,0
kamar,10
tahun_bangun,0
kondisi,0


### Interpretasi

Dataset terdiri dari beberapa atribut yang menggambarkan informasi properti seperti luas bangunan, harga, kota, jumlah kamar, tahun pembangunan, serta kondisi rumah. Sebelum dilakukan analisis lebih lanjut, diperlukan proses data cleaning karena dataset masih mengandung missing values, data duplikat, inkonsistensi penulisan, serta outlier.

# Data Cleaning

Pada tahap ini dilakukan proses pembersihan data yang meliputi penghapusan data duplikat, normalisasi penulisan teks, penanganan missing values, serta penanganan outlier menggunakan metode IQR Fence.

## 1. Menghapus Data Duplikat

In [7]:
# Mengecek jumlah data duplikat sebelum dihapus

print("Jumlah data duplikat sebelum dihapus:", df.duplicated().sum())

# Menghapus data duplikat
df.drop_duplicates(inplace=True)

print("Shape setelah menghapus duplikat:", df.shape)

Jumlah data duplikat sebelum dihapus: 0
Shape setelah menghapus duplikat: (130, 7)


### Interpretasi

Data duplikat dapat menyebabkan hasil analisis menjadi tidak akurat karena satu data dihitung lebih dari satu kali. Oleh karena itu, seluruh data duplikat dihapus sebelum proses analisis dilanjutkan.

## 2. Normalisasi Data String

In [8]:
# Normalisasi penulisan teks

df['kota'] = df['kota'].str.strip().str.title()

df['kondisi'] = df['kondisi'].str.strip().str.lower()

df[['kota','kondisi']].head()

,kota,kondisi
0,Jogja,baik
1,Medan,bagus
2,Depok,baik
3,Ygy,baik
4,Medan,sedang


### Interpretasi

Normalisasi dilakukan agar penulisan data menjadi konsisten. Misalnya, penulisan "jakarta", " Jakarta ", dan "JAKARTA" akan diseragamkan sehingga tidak dianggap sebagai data yang berbeda.

## 3. Mengatasi Missing Values

In [9]:
# Mengisi missing values

df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())

df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())

df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

print(df.isnull().sum())

id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64


### Interpretasi

Nilai yang hilang pada data numerik diisi menggunakan median agar tidak terlalu dipengaruhi oleh outlier. Sementara itu, kolom kategorikal diisi menggunakan modus karena merupakan nilai yang paling sering muncul.

## 4. Menangani Outlier dengan Metode IQR Fence

In [10]:
kolom = ['harga_juta', 'luas_m2', 'tahun_bangun']

for col in kolom:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)

print("Outlier berhasil ditangani.")

Outlier berhasil ditangani.


### Interpretasi

Metode IQR Fence digunakan untuk membatasi nilai yang berada di luar rentang normal. Dengan cara ini, data ekstrem tidak langsung dihapus, tetapi disesuaikan agar tidak memberikan pengaruh yang terlalu besar terhadap hasil analisis.

## 5. Validasi Hasil Data Cleaning

In [11]:
print("Missing Values")
print(df.isnull().sum())

print()

print("Jumlah Duplikat")
print(df.duplicated().sum())

print()

print("Shape Akhir")
print(df.shape)

Missing Values
id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64

Jumlah Duplikat
0

Shape Akhir
(130, 7)


In [21]:
assert df.isnull().sum().sum() == 0, "Masih ada missing value"

assert df.duplicated().sum() == 0, "Masih ada data duplikat"

print("Validasi berhasil.")

Validasi berhasil.


### Interpretasi

Hasil validasi menunjukkan bahwa seluruh missing values telah berhasil ditangani dan tidak terdapat lagi data duplikat pada dataset. Dataset kini berada dalam kondisi yang lebih siap untuk digunakan pada proses analisis berikutnya.

## 6. Menyimpan Dataset Bersih

In [14]:
df.to_csv("housing_clean.csv", index=False)

print("Dataset berhasil disimpan sebagai housing_clean.csv")

Dataset berhasil disimpan sebagai housing_clean.csv


# Mengakses Data dari REST API

Selain menggunakan file CSV, data dalam Data Science juga dapat diperoleh melalui REST API. Pada praktikum ini digunakan API JSONPlaceholder sebagai contoh layanan API publik untuk mengambil data dalam format JSON.

In [20]:
url = "https://jsonplaceholder.typicode.com/posts"

response = requests.get(url)

print("Status Code:", response.status_code)

Status Code: 200


In [16]:
data_api = response.json()

len(data_api)

100

In [17]:
df_api = pd.DataFrame(data_api)

df_api.head()

,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...


In [18]:
df_api.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   userId  100 non-null    int64 
 1   id      100 non-null    int64 
 2   title   100 non-null    object
 3   body    100 non-null    object
dtypes: int64(2), object(2)
memory usage: 3.3+ KB


### Interpretasi

Data berhasil diambil dari REST API JSONPlaceholder dalam format JSON, kemudian diubah menjadi DataFrame menggunakan Pandas. Dari hasil tersebut terlihat bahwa data API dapat diperlakukan sama seperti dataset CSV sehingga dapat dianalisis menggunakan berbagai fungsi yang tersedia pada Pandas.

In [19]:
df_api.to_csv("jsonplaceholder_posts.csv", index=False)

print("Data API berhasil disimpan.")

Data API berhasil disimpan.


# Kesimpulan

Pada praktikum ini saya mempelajari proses data cleaning menggunakan Pandas, mulai dari mendeteksi missing values, menghapus data duplikat, melakukan normalisasi data, menangani missing values dengan metode imputasi, hingga menangani outlier menggunakan metode IQR Fence. Setelah proses pembersihan selesai, dataset divalidasi dan disimpan kembali sebagai dataset yang siap digunakan untuk analisis.

Selain itu, saya juga mempelajari cara mengambil data dari REST API menggunakan library Requests. Data yang diperoleh dalam format JSON dapat diubah menjadi DataFrame sehingga dapat diproses menggunakan Pandas seperti halnya data dari file CSV. Praktikum ini memberikan pemahaman bahwa kualitas data sangat berpengaruh terhadap hasil analisis dan menjadi salah satu tahapan penting dalam proses Data Science.